# Nick Favoriti Project 0 — ccmxpf_lnkhist + 3‑month reporting lag

Same R&D vs No R&D portfolio analysis as Nick’s original, but:
- **CCM:** uses `crsp.ccmxpf_lnkhist` (link history) instead of `ccmxpf_linktable`, with `linkdt`/`linkenddt`.
- **Lag:** uses a **3‑month reporting lag**: fiscal data is used only from `available_date = datadate + 3 months` (point-in-time, no look-ahead).

In [ ]:
import wrds
import pandas as pd
import numpy as np
import statsmodels.api as sm

db = wrds.Connection()

## Compustat + CCM link (ccmxpf_lnkhist)

In [ ]:
comp_sql = """
SELECT
    a.gvkey,
    a.datadate,
    a.xrd,
    b.lpermno AS permno
FROM
    comp.funda a
JOIN
    crsp.ccmxpf_lnkhist b ON a.gvkey = b.gvkey
    AND b.linkdt <= a.datadate
    AND (b.linkenddt >= a.datadate OR b.linkenddt IS NULL)
WHERE
    a.indfmt = 'INDL'
    AND a.datafmt = 'STD'
    AND a.popsrc = 'D'
    AND a.consol = 'C'
    AND a.fyear BETWEEN 1978 AND 2022
    AND a.fic = 'USA'
    AND a.curcd = 'USD'
    AND a.exchg BETWEEN 11 AND 19
    AND (a.sich < 6000 OR a.sich > 6999 OR a.sich IS NULL)
    AND (a.sich != 2834 OR a.sich IS NULL)
    AND b.linktype IN ('LU', 'LC')
    AND b.linkprim IN ('P', 'C')
"""
comp = db.raw_sql(comp_sql)

## CRSP — monthly returns and delisting adjustment

In [ ]:
crsp_sql = """
SELECT
    a.permno,
    a.date,
    a.ret,
    a.prc,
    a.shrout,
    c.dlret
FROM
    crsp.msf a
INNER JOIN crsp.msenames b ON a.permno = b.permno
    AND a.date BETWEEN b.namedt AND b.nameendt
LEFT JOIN crsp.msedelist c ON a.permno = c.permno
    AND date_trunc('month', a.date) = date_trunc('month', c.dlstdt)
WHERE
    a.date BETWEEN '1980-01-01' AND '2023-06-30'
    AND a.ret IS NOT NULL
    AND a.ret >= -1
    AND b.shrcd IN (10, 11)
    AND (b.siccd < 6000 OR b.siccd > 6999)
    AND b.siccd != 2834
    AND b.exchcd IN (1, 2, 3)
"""
crsp = db.raw_sql(crsp_sql)

In [ ]:
crsp['dlret'] = pd.to_numeric(crsp['dlret'], errors='coerce')
crsp['ret'] = np.where(crsp['dlret'].notna(),
                       (1 + crsp['ret']) * (1 + crsp['dlret']) - 1,
                       crsp['ret'])

## Market return and risk-free rate

In [ ]:
mkt = db.raw_sql("""
    SELECT date, vwretd AS mkt_ret
    FROM crsp.msi
    WHERE date BETWEEN '1980-01-01' AND '2023-06-30'
""")

rf = db.raw_sql("""
    SELECT date, rf
    FROM ff.factors_monthly
    WHERE date BETWEEN '1980-01-01' AND '2023-06-30'
""")

## R&D flag and lag (has_rd, rd_flag_lag1); 3‑month available date

In [ ]:
comp['datadate'] = pd.to_datetime(comp['datadate'])
comp['xrd'] = pd.to_numeric(comp['xrd'], errors='coerce')
comp['has_rd'] = ((comp['xrd'].notna()) & (comp['xrd'] > 0)).astype(int)
comp = comp.sort_values(['permno', 'datadate'])
comp['rd_flag_lag1'] = comp.groupby('permno')['has_rd'].shift(1)

# 3‑month reporting lag: use fiscal data only from this date onward
comp['available_date'] = comp['datadate'] + pd.DateOffset(months=3)

# One row per (permno, datadate) if link produced duplicates
comp = comp.drop_duplicates(subset=['permno', 'datadate'], keep='last')

## Market cap (lagged for value weighting)

In [ ]:
crsp['date'] = pd.to_datetime(crsp['date'])
crsp['ret'] = pd.to_numeric(crsp['ret'], errors='coerce')
crsp['prc'] = pd.to_numeric(crsp['prc'], errors='coerce')
crsp['shrout'] = pd.to_numeric(crsp['shrout'], errors='coerce')
crsp['mktcap'] = crsp['prc'].abs() * crsp['shrout']
crsp = crsp.sort_values(['permno', 'date'])
crsp['mktcap_lag'] = crsp.groupby('permno')['mktcap'].shift(1)

## Merge CRSP to Compustat with 3‑month lag (as-of available_date)

In [ ]:
comp_for_merge = comp[['permno', 'available_date', 'datadate', 'rd_flag_lag1']].copy()
comp_for_merge = comp_for_merge.dropna(subset=['rd_flag_lag1'])
comp_for_merge = comp_for_merge.sort_values(['permno', 'available_date'])

crsp_sorted = crsp.sort_values(['permno', 'date'])
merged = pd.merge_asof(
    crsp_sorted,
    comp_for_merge,
    left_on='date',
    right_on='available_date',
    by='permno',
    direction='backward'
)
merged = merged[merged['available_date'].notna()]
merged = merged[merged['date'] >= merged['available_date']]

merged = merged.dropna(subset=['ret', 'mktcap_lag'])
merged = merged[merged['mktcap_lag'] > 0]

## Portfolio returns by month and R&D flag

In [ ]:
merged['year_month'] = merged['date'].dt.to_period('M')

portfolio_returns = []
for (ym, rd), group in merged.groupby(['year_month', 'rd_flag_lag1']):
    ew_ret = group['ret'].mean()
    weights = group['mktcap_lag'] / group['mktcap_lag'].sum()
    vw_ret = (weights * group['ret']).sum()
    portfolio_returns.append({
        'date': group['date'].iloc[0],
        'rd_portfolio': int(rd),
        'ew_ret': ew_ret,
        'vw_ret': vw_ret
    })
portfolio_returns = pd.DataFrame(portfolio_returns)

## Merge market and risk-free rate

In [ ]:
portfolio_returns['date'] = pd.to_datetime(portfolio_returns['date'])
mkt['date'] = pd.to_datetime(mkt['date'])
rf['date'] = pd.to_datetime(rf['date'])
portfolio_returns = portfolio_returns.merge(mkt, on='date', how='left')
portfolio_returns = portfolio_returns.merge(rf, on='date', how='left')

portfolio_returns['ew_ret'] = pd.to_numeric(portfolio_returns['ew_ret'], errors='coerce')
portfolio_returns['vw_ret'] = pd.to_numeric(portfolio_returns['vw_ret'], errors='coerce')
portfolio_returns['mkt_ret'] = pd.to_numeric(portfolio_returns['mkt_ret'], errors='coerce')
portfolio_returns['rf'] = pd.to_numeric(portfolio_returns['rf'], errors='coerce').fillna(0)

## CAPM regression and results

In [ ]:
def run_capm(data, ret_col):
    data = data.dropna(subset=[ret_col, 'mkt_ret', 'rf'])
    y = (data[ret_col] - data['rf']).astype(float).values
    x = (data['mkt_ret'] - data['rf']).astype(float).values
    X = sm.add_constant(x)
    model = sm.OLS(y, X).fit()
    return model.params[0], model.tvalues[0]

rd_data = portfolio_returns[portfolio_returns['rd_portfolio'] == 1].copy()
no_rd_data = portfolio_returns[portfolio_returns['rd_portfolio'] == 0].copy()

rd_ew_alpha, rd_ew_t = run_capm(rd_data, 'ew_ret')
rd_vw_alpha, rd_vw_t = run_capm(rd_data, 'vw_ret')
no_rd_ew_alpha, no_rd_ew_t = run_capm(no_rd_data, 'ew_ret')
no_rd_vw_alpha, no_rd_vw_t = run_capm(no_rd_data, 'vw_ret')

In [ ]:
print(f"{'Portfolio':<20} {'Alpha (Monthly)':<18} {'Alpha (Annual)':<18} {'t-stat':<10}")
print("-" * 70)
print(f"{'R&D (Equal-Wt)':<20} {rd_ew_alpha*100:>6.3f}%{'':11} {rd_ew_alpha*12*100:>6.2f}%{'':11} {rd_ew_t:>6.2f}")
print(f"{'R&D (Value-Wt)':<20} {rd_vw_alpha*100:>6.3f}%{'':11} {rd_vw_alpha*12*100:>6.2f}%{'':11} {rd_vw_t:>6.2f}")
print(f"{'No R&D (Equal-Wt)':<20} {no_rd_ew_alpha*100:>6.3f}%{'':11} {no_rd_ew_alpha*12*100:>6.2f}%{'':11} {no_rd_ew_t:>6.2f}")
print(f"{'No R&D (Value-Wt)':<20} {no_rd_vw_alpha*100:>6.3f}%{'':11} {no_rd_vw_alpha*12*100:>6.2f}%{'':11} {no_rd_vw_t:>6.2f}")

db.close()